In [81]:
import pyspark
from pyspark.sql import SparkSession
from pyspark.sql import types as T
from pyspark.sql import functions as F
from pyspark.sql.window import Window

In [2]:
spark = SparkSession.builder \
            .master("local[*]") \
            .appName('test') \
            .getOrCreate()

your 131072x1 screen size is bogus. expect trouble
25/07/07 12:18:50 WARN Utils: Your hostname, SRCIND-21BQ9G3 resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
25/07/07 12:18:50 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/07/07 12:18:53 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [3]:
# Load Data
movies_path = "./movies.csv"
ratings_path = "./ratings.csv"
tags_path = "./tags.csv"

movies_schema = T.StructType([
    T.StructField("movieId", T.IntegerType(), True),
    T.StructField("title", T.StringType(), True),
    T.StructField("genres", T.StringType(), True)
])

ratings_schema = T.StructType([
    T.StructField("userId", T.IntegerType(), True),
    T.StructField("movieId", T.IntegerType(), True),
    T.StructField("rating", T.FloatType(), True),
    T.StructField("timestamp", T.IntegerType(), True)
])

tags_schema = T.StructType([
    T.StructField("userId", T.IntegerType(), True),
    T.StructField("movieId", T.IntegerType(), True),
    T.StructField("tag", T.StringType(), True),
    T.StructField("timestamp", T.IntegerType(), True)
])

In [4]:
df_movies = spark.read.csv(movies_path, header=True, schema=movies_schema)
df_ratings = spark.read.csv(ratings_path, header=True, schema=ratings_schema)
df_tags = spark.read.csv(tags_path, header=True, schema=tags_schema)

In [5]:
df_movies.show(5)
df_ratings.show(5)
df_tags.show(5)

+-------+--------------------+--------------------+
|movieId|               title|              genres|
+-------+--------------------+--------------------+
|      1|    Toy Story (1995)|Adventure|Animati...|
|      2|      Jumanji (1995)|Adventure|Childre...|
|      3|Grumpier Old Men ...|      Comedy|Romance|
|      4|Waiting to Exhale...|Comedy|Drama|Romance|
|      5|Father of the Bri...|              Comedy|
+-------+--------------------+--------------------+
only showing top 5 rows

+------+-------+------+---------+
|userId|movieId|rating|timestamp|
+------+-------+------+---------+
|     1|      1|   4.0|964982703|
|     1|      3|   4.0|964981247|
|     1|      6|   4.0|964982224|
|     1|     47|   5.0|964983815|
|     1|     50|   5.0|964982931|
+------+-------+------+---------+
only showing top 5 rows

+------+-------+---------------+----------+
|userId|movieId|            tag| timestamp|
+------+-------+---------------+----------+
|     2|  60756|          funny|1445714994|


In [6]:
# extract timestamp from unix timestamp
df_ratings = df_ratings.withColumn("timestamp", F.from_unixtime(df_ratings.timestamp).cast(T.TimestampType()))
df_tags = df_tags.withColumn("timestamp", F.from_unixtime(df_tags.timestamp).cast(T.TimestampType()))

df_ratings.show(5)
df_tags.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-31 00:15:03|
|     1|      3|   4.0|2000-07-30 23:50:47|
|     1|      6|   4.0|2000-07-31 00:07:04|
|     1|     47|   5.0|2000-07-31 00:33:35|
|     1|     50|   5.0|2000-07-31 00:18:51|
+------+-------+------+-------------------+
only showing top 5 rows

+------+-------+---------------+-------------------+
|userId|movieId|            tag|          timestamp|
+------+-------+---------------+-------------------+
|     2|  60756|          funny|2015-10-25 00:59:54|
|     2|  60756|Highly quotable|2015-10-25 00:59:56|
|     2|  60756|   will ferrell|2015-10-25 00:59:52|
|     2|  89774|   Boxing story|2015-10-25 01:03:27|
|     2|  89774|            MMA|2015-10-25 01:03:20|
+------+-------+---------------+-------------------+
only showing top 5 rows



In [7]:
df_movies.printSchema()
df_ratings.printSchema()
df_tags.printSchema()

root
 |-- movieId: integer (nullable = true)
 |-- title: string (nullable = true)
 |-- genres: string (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- rating: float (nullable = true)
 |-- timestamp: timestamp (nullable = true)

root
 |-- userId: integer (nullable = true)
 |-- movieId: integer (nullable = true)
 |-- tag: string (nullable = true)
 |-- timestamp: timestamp (nullable = true)



In [8]:
# Work with spark SQL

df_movies.createOrReplaceTempView("movies")
df_ratings.createOrReplaceTempView("ratings")
df_tags.createOrReplaceTempView("tags")

Q1. Show the aggregated number of ratings per year

In [9]:
query= """
        SELECT 
            YEAR(timestamp) as year,
            COUNT(rating) as ratings 
        FROM ratings 
        GROUP BY 1 
        ORDER BY YEAR(timestamp) desc
"""

output = spark.sql(query)
output.show(5)

+----+-------+
|year|ratings|
+----+-------+
|2018|   6418|
|2017|   8199|
|2016|   6702|
|2015|   6616|
|2014|   1439|
+----+-------+
only showing top 5 rows



In [10]:
# Write data into single file
output.coalesce(1).write.mode("overwrite").format('csv').option('header', 'true') .option('delimiter', ',').save('./results/agg_ratings.csv')
print("Write Successfull")

Write Successfull


Q2. Show the average monthly number of rating

In [11]:
query = """
SELECT *
FROM ratings
LIMIT 5
"""

output = spark.sql(query)
output.show(5)

+------+-------+------+-------------------+
|userId|movieId|rating|          timestamp|
+------+-------+------+-------------------+
|     1|      1|   4.0|2000-07-31 00:15:03|
|     1|      3|   4.0|2000-07-30 23:50:47|
|     1|      6|   4.0|2000-07-31 00:07:04|
|     1|     47|   5.0|2000-07-31 00:33:35|
|     1|     50|   5.0|2000-07-31 00:18:51|
+------+-------+------+-------------------+



In [12]:
query = """
SELECT 
    MONTH(timestamp) as month,
    AVG(rating) as avg_rating
FROM ratings
GROUP BY 1
"""

output = spark.sql(query)
output.show(5)

+-----+------------------+
|month|        avg_rating|
+-----+------------------+
|   12|3.5212425432853194|
|    1|3.5031358885017423|
|    6| 3.420714527217401|
|    3| 3.455664194200944|
|    5|3.4466414118081863|
+-----+------------------+
only showing top 5 rows



In [13]:
query = """
SELECT 
    LEFT(timestamp, 7) as month,
    AVG(rating) as avg_rating
FROM ratings
GROUP BY 1
"""

output = spark.sql(query)
output.show(5)

+-------+------------------+
|  month|        avg_rating|
+-------+------------------+
|1999-10|3.6920700308959837|
|2013-05| 3.982889733840304|
|2009-07| 3.949685534591195|
|1999-11|3.9272727272727272|
|2002-11|             3.125|
+-------+------------------+
only showing top 5 rows



In [14]:
# Write data into single file
output.coalesce(1).write.mode("overwrite").format('csv').option('header', 'true') .option('delimiter', ',').save('./results/avg_monthly_ratings.csv')
print("Write Successfull")

Write Successfull


Q3. Show the rating levels distribution

In [15]:
query = """
WITH rating_bucket AS (
    SELECT
        rating,
        CASE
            WHEN rating < 2 THEN '0-2'
            WHEN rating >= 2 AND rating < 4 THEN '2-4'
            ELSE '> 4'
        END as bucket
    FROM ratings r
), rating_counts AS (
    SELECT
        bucket,
        COUNT(*) as count
    FROM rating_bucket
    GROUP BY bucket
    ORDER BY bucket
)
SELECT
    bucket,
    count,
    ROUND((count/ SUM(count) OVER ()) * 100, 2) as percentage
FROM rating_counts

"""
output = spark.sql(query)
output.show()

25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:48 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 1

+------+-----+----------+
|bucket|count|percentage|
+------+-----+----------+
|   0-2| 5972|      5.92|
|   2-4|46284|      45.9|
|   > 4|48580|     48.18|
+------+-----+----------+



In [16]:
# Write data in HDFS into single file
output.coalesce(1).write.mode("overwrite").format('csv').option('header', 'true') .option('delimiter', ',').save('./results/distribution_ratings.csv')
print("Write Successfull")

25/07/07 12:19:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:49 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 12:19:50 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 1

Write Successfull


Q4. Show the 18 movies that are tagged but not rated  

In [17]:
query= """
with t1 as (
Select distinct 
    t.movieID 
FROM TAGS as t
left join RATINGS as r
    on t.movieID=r.movieID
where r.movieID IS NULL
)
Select m.title 
from MOVIES as m
inner join t1
on m.movieID=t1.movieID
order by 1
"""
          

output = spark.sql(query)
print("Movies with tags but no rating: ", output.count())
output.show(truncate=False)

Movies with tags but no rating:  18


+--------------------------------------------+
|title                                       |
+--------------------------------------------+
|Browning Version, The (1951)                |
|Call Northside 777 (1948)                   |
|Chalet Girl (2011)                          |
|Chosen, The (1981)                          |
|Color of Paradise, The (Rang-e khoda) (1999)|
|For All Mankind (1989)                      |
|I Know Where I'm Going! (1945)              |
|In the Realms of the Unreal (2004)          |
|Innocents, The (1961)                       |
|Mutiny on the Bounty (1962)                 |
|Niagara (1953)                              |
|Parallax View, The (1974)                   |
|Proof (1991)                                |
|Road Home, The (Wo de fu qin mu qin) (1999) |
|Roaring Twenties, The (1939)                |
|Scrooge (1970)                              |
|This Gun for Hire (1942)                    |
|Twentieth Century (1934)                    |
+------------

Q5. Show the movies that have rating but no tag

In [18]:
query= """
with t1 as (
Select distinct 
    t.movieID 
from TAGS as t
left join RATINGS as r
on t.movieID=r.movieID
where r.movieID IS NULL
)
Select m.title 
from MOVIES as m
inner join t1
on m.movieID=t1.movieID
order by 1
"""
          
print("Movies with tags but no rating: ", output.count())
output = spark.sql(query)
output.show(truncate=False)

Movies with tags but no rating:  18


+--------------------------------------------+
|title                                       |
+--------------------------------------------+
|Browning Version, The (1951)                |
|Call Northside 777 (1948)                   |
|Chalet Girl (2011)                          |
|Chosen, The (1981)                          |
|Color of Paradise, The (Rang-e khoda) (1999)|
|For All Mankind (1989)                      |
|I Know Where I'm Going! (1945)              |
|In the Realms of the Unreal (2004)          |
|Innocents, The (1961)                       |
|Mutiny on the Bounty (1962)                 |
|Niagara (1953)                              |
|Parallax View, The (1974)                   |
|Proof (1991)                                |
|Road Home, The (Wo de fu qin mu qin) (1999) |
|Roaring Twenties, The (1939)                |
|Scrooge (1970)                              |
|This Gun for Hire (1942)                    |
|Twentieth Century (1934)                    |
+------------

Q6. Focusing on the rated untagged movies with more than 30 user ratings,
show the top 10 movies in terms of average rating and number of
ratings

In [40]:
# Get the rated untagged moview 
# filter out movies with more than 30 user ratings
# show the top 10 moviews in terms of average rating and number of ratings

query= """
with avg_rating_data as (
    SELECT 
        r.movieId,
        m.title,
        ROUND(AVG(r.rating), 2) as avg_rating,
        DENSE_RANK() OVER (ORDER BY ROUND(AVG(r.rating), 2) DESC) as rank,
        COUNT(r.userId) as num_ratings
    FROM ratings r
    LEFT JOIN tags t
    ON r.movieId = t.movieId
    JOIN movies m
        ON r.movieId = m.movieId
    WHERE t.movieId IS NULL
    GROUP BY r.movieId, m.title
    HAVING COUNT(r.userId) > 30
    ORDER BY avg_rating DESC
), num_rating_data as (
    SELECT 
        r.movieId,
        m.title,
        ROUND(AVG(r.rating), 2) as avg_rating,
        COUNT(r.userId) as num_ratings,
        DENSE_RANK() OVER (ORDER BY COUNT(r.userId) DESC) as rank
    FROM ratings r
    LEFT JOIN tags t
    ON r.movieId = t.movieId
    JOIN movies m
        ON r.movieId = m.movieId
    WHERE t.movieId IS NULL
    GROUP BY r.movieId, m.title
    HAVING COUNT(r.userId) > 30
    ORDER BY num_ratings DESC
)
SELECT
    a.movieId,
    a.title,
    a.avg_rating,
    a.num_ratings,
    a.rank as avg_rating_rank,
    r.movieId as num_rating_movieId,
    r.title as num_rating_title,
    r.avg_rating as num_rating_avg_rating,
    r.num_ratings as num_rating_num_ratings,
    r.rank as num_rating_rank
FROM avg_rating_data a
JOIN num_rating_data r
 ON a.rank = r.rank
WHERE a.rank <= 10 AND r.rank <= 10
"""
          

output = spark.sql(query)
output.show(truncate=False)

25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 13:54:29 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 1

+-------+----------------------------------------------------------+----------+-----------+---------------+------------------+--------------------------------------------+---------------------+----------------------+---------------+
|movieId|title                                                     |avg_rating|num_ratings|avg_rating_rank|num_rating_movieId|num_rating_title                            |num_rating_avg_rating|num_rating_num_ratings|num_rating_rank|
+-------+----------------------------------------------------------+----------+-----------+---------------+------------------+--------------------------------------------+---------------------+----------------------+---------------+
|3275   |Boondock Saints, The (2000)                               |4.22      |43         |1              |2858              |American Beauty (1999)                      |4.06                 |204                   |1              |
|1199   |Brazil (1985)                                             |

Q7. What is the average number of tags per movie in tags DF?  
And the average number of tags per user?  
How does it compare with the average number of tags a user assigns to a movie?

In [65]:
query = """
with avg_tags_per_movie as (
    SELECT 
        ROUND(COUNT(tag) * 1.0 / COUNT(DISTINCT movieId), 2) AS avg_tags_per_movie
    FROM tags
), avg_tags_per_user as (
    SELECT 
        ROUND(COUNT(tag) * 1.0 / COUNT(DISTINCT userId), 2) AS avg_tags_per_user
    FROM tags
)
SELECT 
    a.avg_tags_per_movie,
    b.avg_tags_per_user,
    CASE
        WHEN a.avg_tags_per_movie > b.avg_tags_per_user THEN 'More tags per movie than per user'
        WHEN a.avg_tags_per_movie < b.avg_tags_per_user THEN 'More tags per user than per movie'
        ELSE 'Equal tags per movie and per user'
    END AS comparison
FROM avg_tags_per_movie a, avg_tags_per_user b
"""

output = spark.sql(query)
output.show(truncate=False)

+------------------+-----------------+---------------------------------+
|avg_tags_per_movie|avg_tags_per_user|comparison                       |
+------------------+-----------------+---------------------------------+
|2.34              |63.50            |More tags per user than per movie|
+------------------+-----------------+---------------------------------+



Q8. Identify the users that tagged movies without rating them

In [68]:
query = """
SELECT DISTINCT
    t.userId
FROM tags t 
LEFT JOIN ratings r 
    ON t.movieId = r.movieId
WHERE r.rating IS NULL
"""

output = spark.sql(query)
output.show(truncate=False)

+------+
|userId|
+------+
|474   |
|318   |
|543   |
|288   |
+------+



Q9. What is the average number of ratings per user in ratings DF? And the
average number of ratings per movie?

In [74]:
query = """
WITH avg_user_ratings AS (
    SELECT 
       ROUND(COUNT(*) / COUNT(DISTINCT userId), 2) AS avg_ratings_per_user
    FROM ratings r
), avg_movie_ratings AS (
    SELECT 
        ROUND(COUNT(*) / COUNT(DISTINCT movieId), 2) AS avg_ratings_per_movie
    FROM ratings r
)

SELECT 
    avg_user_ratings.avg_ratings_per_user,
    avg_movie_ratings.avg_ratings_per_movie
FROM avg_user_ratings, avg_movie_ratings

"""

output = spark.sql(query)
output.show(truncate=False)

+--------------------+---------------------+
|avg_ratings_per_user|avg_ratings_per_movie|
+--------------------+---------------------+
|165.3               |10.37                |
+--------------------+---------------------+



Q10. What is the predominant (frequency based) genre per rating level?

In [84]:
# Join ratings and movies
df_joined = df_ratings.join(df_movies, on="movieId")

# Split and explode genres
df_exploded = df_joined.withColumn("genre", F.explode(F.split("genres", "\\|")))

df_exploded.select("userId", "movieId", "rating", "title", "genre").show(5, truncate=False)

df_counts = df_exploded.groupBy("rating", "genre").count()

# Window to rank genres by count within each rating
window = Window.partitionBy("rating").orderBy(F.col("count").desc())

# Add row number and filter for top genre per rating
df_top_genre = df_counts.withColumn("rn", F.row_number().over(window)).filter(F.col("rn") == 1)

df_top_genre.show()

+------+-------+------+----------------+---------+
|userId|movieId|rating|title           |genre    |
+------+-------+------+----------------+---------+
|1     |1      |4.0   |Toy Story (1995)|Adventure|
|1     |1      |4.0   |Toy Story (1995)|Animation|
|1     |1      |4.0   |Toy Story (1995)|Children |
|1     |1      |4.0   |Toy Story (1995)|Comedy   |
|1     |1      |4.0   |Toy Story (1995)|Fantasy  |
+------+-------+------+----------------+---------+
only showing top 5 rows



+------+------+-----+---+
|rating| genre|count| rn|
+------+------+-----+---+
|   0.5|Comedy|  632|  1|
|   1.0|Comedy| 1317|  1|
|   1.5|Comedy|  895|  1|
|   2.0|Comedy| 3405|  1|
|   2.5|Comedy| 2530|  1|
|   3.0|Comedy| 8306|  1|
|   3.5| Drama| 5514|  1|
|   4.0| Drama|12360|  1|
|   4.5| Drama| 4217|  1|
|   5.0| Drama| 6350|  1|
+------+------+-----+---+



In [85]:
df_tags.show(5)

+------+-------+---------------+-------------------+
|userId|movieId|            tag|          timestamp|
+------+-------+---------------+-------------------+
|     2|  60756|          funny|2015-10-25 00:59:54|
|     2|  60756|Highly quotable|2015-10-25 00:59:56|
|     2|  60756|   will ferrell|2015-10-25 00:59:52|
|     2|  89774|   Boxing story|2015-10-25 01:03:27|
|     2|  89774|            MMA|2015-10-25 01:03:20|
+------+-------+---------------+-------------------+
only showing top 5 rows



Q11. What is the predominant tag per genre and the most tagged genres?

In [93]:
# Join tags and movies
df_joined = df_tags.join(df_movies, on="movieId")

# Split and explode genres
df_exploded = df_joined.withColumn("genre", F.explode(F.split("genres", "\\|")))

df_exploded.select("userId", "movieId", "title", "genre", "tag").show(5, truncate=False)

df_counts = df_exploded.groupBy("genre", "tag").count()
df_counts.show(5)

# Window to get predominant tag per genre
window = Window.partitionBy("genre").orderBy(F.col("count").desc())

# Add row number and filter for top genre per tag
df_top_genre = df_counts.withColumn("rn", F.row_number().over(window)).filter(F.col("rn") == 1)

print("Predominant tag per genre: ")
df_top_genre.show()

# Most tagged genres
print("Most tagged genres: ")
df_exploded.groupBy("genre").count().orderBy(F.col("count").desc()).show(10, truncate=False)

+------+-------+----------------+---------+---+
|userId|movieId|title           |genre    |tag|
+------+-------+----------------+---------+---+
|567   |1      |Toy Story (1995)|Adventure|fun|
|567   |1      |Toy Story (1995)|Animation|fun|
|567   |1      |Toy Story (1995)|Children |fun|
|567   |1      |Toy Story (1995)|Comedy   |fun|
|567   |1      |Toy Story (1995)|Fantasy  |fun|
+------+-------+----------------+---------+---+
only showing top 5 rows

+--------+------------------+-----+
|   genre|               tag|count|
+--------+------------------+-----+
|Thriller|             crime|    8|
|Thriller|        Palme d'Or|    1|
|Thriller|            ironic|    2|
|   Crime|          dialogue|    1|
|  Comedy|big boys with guns|    1|
+--------+------------------+-----+
only showing top 5 rows

Predominant tag per genre: 
+------------------+------------------+-----+---+
|             genre|               tag|count| rn|
+------------------+------------------+-----+---+
|(no genres list

Q12. What are the most predominant (popularity based) movies?

In [97]:
query = """
SELECT
    m.movieId,
    m.title,
    COUNT(*) as num_ratings
FROM ratings r
JOIN movies m
    ON r.movieId = m.movieId
GROUP BY m.movieId, m.title
ORDER BY num_ratings DESC
LIMIT 10
"""

output = spark.sql(query)
output.show(truncate=False)

+-------+-----------------------------------------+-----------+
|movieId|title                                    |num_ratings|
+-------+-----------------------------------------+-----------+
|356    |Forrest Gump (1994)                      |329        |
|318    |Shawshank Redemption, The (1994)         |317        |
|296    |Pulp Fiction (1994)                      |307        |
|593    |Silence of the Lambs, The (1991)         |279        |
|2571   |Matrix, The (1999)                       |278        |
|260    |Star Wars: Episode IV - A New Hope (1977)|251        |
|480    |Jurassic Park (1993)                     |238        |
|110    |Braveheart (1995)                        |237        |
|589    |Terminator 2: Judgment Day (1991)        |224        |
|527    |Schindler's List (1993)                  |220        |
+-------+-----------------------------------------+-----------+



Q13. Top 10 movies in terms of average rating (provided more than 30 users reviewed them)

In [ ]:
# movies with more than 30 user ratings
# filter the top 10 movies in terms of average rating
query = """
WITH movie_ratings AS (
    SELECT 
        r.movieId,
        COUNT(DISTINCT r.userId) as num_users
    FROM ratings r
    GROUP BY r.movieId
    HAVING COUNT(DISTINCT r.userId) > 30
), ranking_data as (
    SELECT
        mr.movieId,
        m.title,
        ROUND(AVG(r.rating), 2) as avg_rating,
        DENSE_RANK() OVER(ORDER BY ROUND(AVG(r.rating), 2) DESC) as rank
    FROM ratings r
    JOIN movie_ratings mr
        ON r.movieId = mr.movieId
    JOIN movies m
        ON m.movieId = mr.movieId
    GROUP BY mr.movieId, m.title
    ORDER BY avg_rating DESC
)
SELECT *
FROM ranking_data rd
WHERE rd.rank <= 10
"""

output = spark.sql(query)
output.show(30, truncate=False)

25/07/07 16:34:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 16:34:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 16:34:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 16:34:34 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 16:34:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 16:34:35 WARN WindowExec: No Partition Defined for Window operation! Moving all data to a single partition, this can cause serious performance degradation.
25/07/07 1

+--------------------+-----------+------+
|               title| avg_rating|ranker|
+--------------------+-----------+------+
|Shawshank Redempt...|4.429022082|     1|
|Lawrence of Arabi...|        4.3|     2|
|Godfather, The (1...|  4.2890625|     3|
|   Fight Club (1999)| 4.27293578|     4|
|Cool Hand Luke (1...|4.271929825|     5|
|Dr. Strangelove o...|4.268041237|     6|
|  Rear Window (1954)|4.261904762|     7|
|Godfather: Part I...|4.259689922|     8|
|Departed, The (2006)|4.252336449|     9|
|   Goodfellas (1990)|       4.25|    10|
+--------------------+-----------+------+

